In [1]:
from torch_geometric.data import HeteroData
import torch
import pickle
import numpy as np
import torch

In [2]:
with open("D:\\Entity Aspect Linking\\data\picklefiles\\final_eal.pkl", 'rb') as eal:
    data = pickle.load(eal)

In [26]:
ent = [data[i][0] for i in range(len(data))]

In [27]:
ent[1]

{'id': '1',
 'target_entity': 'Gautama Buddha',
 'paragraph': "The iconography of Gautama Buddha in Laos and Thailand recall specific episodes during his travels and teachings that are familiar to the Buddhists according to an iconography with specific rules. The Buddha is always represented with certain physical attributes, and in specified dress and specified poses. Each pose, and particularly the position and gestures of the Buddha's hands, has a defined meaning which is familiar to Buddhists.  In other Buddhist countries, different but related iconography is used, for example the mudras in Indian art. Certain ones of these are considered particularly auspicious for those born on particular days of the week.",
 'entities': [{'eid': '10',
   'entity': 'Gautama Buddha',
   'mention': 'Gautama Buddha'},
  {'eid': '11', 'entity': 'Laos', 'mention': 'Laos'},
  {'eid': '12', 'entity': 'Thailand', 'mention': 'Thailand'},
  {'eid': '13', 'entity': 'Episode', 'mention': 'episode'},
  {'eid':

In [4]:
asp = [data[i][1] for i in range(len(data))]

In [5]:
count = 0
for e in ent:
    count += len(e['entities'])

In [6]:
count

467

In [7]:
graph = HeteroData()

In [8]:
len(ent)

100

In [9]:
def get_node_id(data, subject = "a_entities"):
    res = []
    dic = {}
    if subject == "target_entity":
        for item in data:
            target, _ = item
            res.append(target['id'])
        return list(map(int,res))
    elif subject == "t_entities":
        i = 0
        for item in data:
            target,_ = item
            for ent in target['entities']:
                dic[ent['eid']] = i
                i += 1
        return dic
    elif subject == "aspect_entity":
        i = 0
        for item in data:
            _, target = item
            dic[target["id"]] = i
            i+=1
            for l in target["candidate_aspects"]:
                if l["aspect_name"] != target["true_aspect"]:
                    dic[l["id"]] = i
                    i+=1           
        return dic
    elif subject == 'a_entities':
        i = 0
        for item in data:
            _, target = item
            for l in target["candidate_aspects"]:
                for el in l["entities"]:
                    dic[el['eid']] = i
                    i+= 1
        return dic
        
    

In [10]:
t_key = get_node_id(data, subject = "t_entities")
tid = list(get_node_id(data, subject = "t_entities").values())

In [11]:
def get_edge_entities(data, key , type = 'target_entity', **kwargs):
    edge = []
    if type == 'target_entity':
        for item in data:
            ent, _ = item
            ent_id = int(ent['id'])
            for item in ent['entities']:
                edge.append([ent_id, key[item['eid']]])
        return np.array(edge).T

    elif type =='aspect_entity':
        for item in data:
            _, asp = item
            asp_id = asp['id']
            #print(asp_id)
            for cand in asp['candidate_aspects']:
                if cand['aspect_name'] == asp['true_aspect']:
                    for e in cand['entities']:
                        edge.append([aspdict[asp_id], key[e['eid']]])
        return np.array(edge).T
                    
    


In [12]:
def get_target_edges(data, asp_dict):
    edge = []
    for item in data:
        ent, asp = item
        tid = int(ent['id'])
        aspid = asp_dict[asp['id']]
        edge.append([tid, aspid])
    return np.array(edge).T

In [13]:
ent_edge_index = torch.tensor(get_edge_entities(data, t_key))

In [14]:
aspdict = get_node_id(data,subject = "aspect_entity")
asp_id = list(aspdict.values())

In [15]:
aspdict

{'0': 0,
 'A00': 1,
 'A01': 2,
 'A03': 3,
 'A04': 4,
 'A05': 5,
 'A06': 6,
 '1': 7,
 'A10': 8,
 'A11': 9,
 'A13': 10,
 'A14': 11,
 'A15': 12,
 'A16': 13,
 '2': 14,
 'A20': 15,
 'A21': 16,
 'A23': 17,
 'A24': 18,
 'A25': 19,
 'A26': 20,
 '3': 21,
 'A30': 22,
 'A31': 23,
 'A33': 24,
 'A34': 25,
 'A35': 26,
 'A36': 27,
 '4': 28,
 'A40': 29,
 'A41': 30,
 'A43': 31,
 'A44': 32,
 'A45': 33,
 'A46': 34,
 '5': 35,
 'A50': 36,
 'A51': 37,
 'A53': 38,
 'A54': 39,
 'A55': 40,
 'A56': 41,
 '6': 42,
 'A60': 43,
 'A62': 44,
 'A63': 45,
 'A64': 46,
 'A65': 47,
 'A66': 48,
 '7': 49,
 'A71': 50,
 'A72': 51,
 'A73': 52,
 'A74': 53,
 'A75': 54,
 'A76': 55,
 '8': 56,
 'A80': 57,
 'A81': 58,
 'A83': 59,
 'A84': 60,
 'A85': 61,
 'A86': 62,
 '9': 63,
 'A90': 64,
 'A91': 65,
 'A93': 66,
 'A94': 67,
 'A95': 68,
 'A96': 69,
 '10': 70,
 'A100': 71,
 'A101': 72,
 'A103': 73,
 '11': 74,
 'A110': 75,
 'A111': 76,
 'A113': 77,
 '12': 78,
 'A120': 79,
 'A121': 80,
 'A123': 81,
 '13': 82,
 'A130': 83,
 'A131': 84,
 'A

In [16]:
a_entdict = get_node_id(data, subject = "a_entities")
a_ent_id = list(a_entdict.values())

In [17]:
len(list(a_entdict.keys()))

18252

In [18]:
asp_edge_index = torch.tensor(get_edge_entities(data, key = a_entdict, type = 'aspect_entity', a_entdict = a_entdict))

In [19]:
target_edge_index = torch.tensor(get_target_edges(data, aspdict))

In [20]:
PTH = 'picklefiles'
def read_pickle(filename):
    with open(f'{PTH}\\{filename}', 'rb') as f:
        emb = np.load(f, allow_pickle = True)
    return emb

def read_tensor(filename):
    return torch.load(f'{PTH}\\{filename}')

In [21]:
ent1 = read_tensor('EntityEmb.T')
asp = read_tensor('AspectEmb.T')
tent = read_pickle('emb_targetentities.pkl')
aspent = read_pickle('emb_aspectentities.pkl')

In [22]:
#print(ent.shape, asp.shape)
asp.shape

torch.Size([100, 384])

In [23]:
graph['target_entity'].num_nodes = len(ent)
graph['target_entity'].node_id = get_node_id(data, subject = 'target_entity')
graph['target_entity'].x = tent
graph['target_entity'].y = ent1
graph['t_entities'].num_nodes = count
graph['t_entities'].node_id = tid
graph['target_entity', 'associated_with', 't_entities'] = ent_edge_index

graph["aspect_entity"].num_nodes = len(aspdict)
graph["aspect_entity"].node_id = asp_id
graph["aspect_entity"].x = aspent
graph["aspect_entity"].y = asp 
graph['a_entities'].num_nodes = len(a_ent_id)
graph['a_entities'].node_id = a_ent_id
graph['aspect_entity', 'associated_with', 'a_entities'] = asp_edge_index

graph['target_entity', 'linked_to', 'aspect_entity'] = target_edge_index


In [24]:
graph

HeteroData(
  (target_entity, associated_with, t_entities)=[2, 467],
  (aspect_entity, associated_with, a_entities)=[2, 8922],
  (target_entity, linked_to, aspect_entity)=[2, 100],
  target_entity={
    num_nodes=100,
    node_id=[100],
    x=[100, 100],
    y=[100, 384]
  },
  t_entities={
    num_nodes=467,
    node_id=[467]
  },
  aspect_entity={
    num_nodes=640,
    node_id=[640]
    num_nodes=639,
    node_id=[639],
    x=[639, 100],
    y=[100, 384]
  },
  a_entities={
    num_nodes=17642,
    node_id=[17642]
  }
)